# Exploring a MIKE DFSU results file with pyramids-mesh

**DFSU** is DHI's proprietary binary container for MIKE 21/3 flexible-mesh *results*: an unstructured mesh (mixed triangles and quads) carrying time-varying solver fields such as surface elevation, water depth and velocity. pyramids-mesh reads it through the `dhi` driver, which uses DHI's `mikeio` library under the hood, and hands back a standard `MeshDataset`.

Because `mikeio` is an optional dependency, the DFSU driver only works when it is installed - via the `pyramids-mesh[dhi]` extra (the project's `dhi` environment). Everything *after* the read is ordinary pyramids: once the mesh is in hand, no DFSU-specific code is involved.

The file here is a real **Øresund** hydrodynamic run - the strait between Denmark and Sweden - so this is a genuine results dataset, not bare geometry.

## Setup

In [ ]:
from pathlib import Path
import numpy as np
from pyramids.netcdf.ugrid import MeshDataset

# Paths are relative to THIS notebook (docs/examples/real-data/), never the repo root.
# read_file takes a pathlib.Path - pass Path objects, never str().
DATA = Path("../../../examples/data/real-data")
DATA.resolve()

In [ ]:
import matplotlib.pyplot as plt
from cleopatra.basemap.tiles import add_tiles

## What is on disk

A single DFSU file holds the whole run - mesh plus every stored timestep of every field.

In [ ]:
for p in sorted((DATA / "dfsu").iterdir()):
    print(p.name, f"({p.stat().st_size:,} bytes)")

## Read the file

Pass `driver="dfsu"`. The driver opens the binary container through `mikeio`, rebuilds the flexible mesh, and attaches each solver field as a data variable. The result is an unstructured 2D `MeshDataset`.

In [ ]:
ds = MeshDataset.read_file(DATA / "dfsu" / "oresundHD_run1.dfsu", driver="dfsu")
ds

The repr confirms a 2D mesh with **2046 nodes** and **3612 faces**, and lists five variables: the four solver fields plus the static bed level `node_z`.

## Let cleopatra pick the plotting backend

cleopatra switches matplotlib to a headless `Agg` backend on the first `read_file`. Calling its chooser once, right after the first read, selects the inline backend in a notebook (and correctly stays on `Agg` in a plain script or CI), so every figure below embeds in the cell output with no manual rasterising.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## Explore the mesh

The scalar properties describe the grid at a glance: node and face counts, its coordinate reference system, and its spatial extent.

In [ ]:
print("n_node:", ds.n_node)
print("n_face:", ds.n_face)
print("crs:   ", ds.crs)
print("epsg:  ", ds.epsg)
print("bounds:", ds.bounds)  # (xmin, ymin, xmax, ymax)

`crs`/`epsg` come back `None`: MIKE stores the projection in a companion file, so the DFSU itself carries no embedded CRS and pyramids reports the geometry faithfully rather than guessing. The `bounds` are nonetheless a projected UTM grid in metres - the ~55 km x ~95 km envelope is the Øresund model domain. Every tile overlay below is guarded by `if ds.epsg:`, so it simply no-ops here and still works unchanged on a georeferenced mesh.

## Tour the solver fields

`data_variable_names` lists every field riding on the mesh. These are the real model results - the reason a DFSU is worth reading rather than just a mesh file.

In [ ]:
ds.data_variable_names

`Surface elevation`, `Total water depth`, `U velocity` and `V velocity` are the time-varying hydrodynamic outputs; `node_z` is the static bathymetry (bed level) at each node.

`summary()` returns a tidy `pandas.DataFrame` of per-variable statistics - a quick sanity check on magnitudes and sign.

In [ ]:
ds.summary()

The solver fields each report a `count` of 18060 = 3612 faces x 5 timesteps, so they are **time-varying** `(time, face)` arrays. `node_z` reports 2046 = one value per node, confirming it is static. Depths run from ~0.2 m in the shallows to ~33 m in the deep channel, and the velocity components straddle zero as the tidal flow reverses - physically sensible for the strait.

## Wireframe - the bare mesh

`plot_outline()` draws the mesh edges with no data colouring. It is the fastest way to see the grid's shape and how resolution is distributed before mapping any results.

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The outline traces the Øresund strait, with the mesh refined around the coastlines and narrows where the flow gradients are steepest.

## Selecting a timestep

The solver fields are `(time, face)`, so a single map needs one timestep. `time_values` lists the stored times (seconds since the start - here five daily snapshots), and `isel(time=...)` returns a new `MeshDataset` holding just that step, ready to plot.

In [ ]:
print("n timesteps:", len(ds.time_values))
print("times (s):   ", list(ds.time_values))
step = ds.isel(time=2)  # a mid-run snapshot

## Mapping the real results

`plot(variable, ...)` colours each face by that field's value. We map three of the four solver fields at the selected step; the returned `cleopatra` `MeshGlyph` wraps the matplotlib figure, which the inline backend flushes into the cell.

### Total water depth

Depth is the clearest picture of the domain's bathymetry *as the model sees it* - deep in the central channel, shallow along the shores.

In [ ]:
_g = step.plot("Total water depth", cmap="viridis", colorbar=True, title="Total water depth (step 2)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The deep navigation channel down the middle of the strait stands out in yellow-green, while the broad shallow banks to either side sit near zero - exactly the structure that steers the tidal flow.

### Surface elevation

Surface elevation is the tidal water level relative to datum. Across the domain it varies only centimetres to tens of centimetres, so a sequential map reveals the gentle set-up/set-down gradient driving the flow.

In [ ]:
_g = step.plot("Surface elevation", cmap="cividis", colorbar=True, title="Surface elevation (step 2)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The water level grades smoothly from one end of the strait to the other - a small but coherent slope. It is this head difference between the Kattegat and the Baltic that pushes water through the Øresund.

### U velocity

`U velocity` is the east-west component of the depth-averaged current. A diverging colormap centred on zero shows eastward (positive) and westward (negative) flow at once.

In [ ]:
_g = step.plot("U velocity", cmap="coolwarm", colorbar=True, title="U velocity (step 2)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The strongest currents concentrate in the deep channel and the narrows, where the cross-section pinches and the flow accelerates; the shallow flanks stay near still. The red/blue split marks where the current turns across the strait.

## Bathymetry from `node_z`

`node_z` is the static bed level carried on the mesh nodes - no timestep needed. Plotting it straight from the full dataset gives the model's bathymetry.

In [ ]:
_g = ds.plot("node_z", cmap="terrain", colorbar=True, title="Bed level (node_z)")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

Negative values are depth below datum: the dark trench is the deep channel (down to ~-37 m), the pale fringe the shallow coastal shelf. This fixed bathymetry is what the time-varying depth field is layered on top of.

## GIS for free

A `MeshDataset` is not a dead end - it converts into the vector types the rest of a GIS workflow expects, and crops like any spatial dataset. Everything below is reached through pyramids; this package only supplied the DFSU driver that built the mesh.

### Faces as vector polygons

`to_geodataframe(variable_name=...)` turns every face into a polygon row carrying that variable's value - a standard GeoPandas `GeoDataFrame` you can join, filter, or export.

In [ ]:
gdf = step.to_geodataframe(variable_name="Total water depth")
print("rows:", len(gdf), " columns:", list(gdf.columns), " crs:", gdf.crs)
gdf.head()

One row per face (3612), with the depth value and a polygon geometry. `crs` is `None`, mirroring the mesh - assign one here if you need to reproject or join against georeferenced data.

### The mesh outline as one polygon

`bounding_polygon()` returns a single shapely polygon tracing the mesh's outer boundary - the model's wetted footprint, useful as a clip mask or extent reference.

In [ ]:
poly = ds.bounding_polygon()
import geopandas as gpd
ax = gpd.GeoSeries([poly]).plot(facecolor="none", edgecolor="crimson", figsize=(5, 6))
if ds.epsg:
    add_tiles(ax, crs=ds.epsg)
plt.title("Mesh bounding polygon")

The single polygon is the dissolved outline of all 3612 faces - the exact extent the model solves over.

### Cropping to a sub-region

`subset_by_bounds(xmin, ymin, xmax, ymax)` returns a new, smaller `MeshDataset` holding only the faces inside the box - the same object type, so every operation above still applies. We crop the selected timestep to the lower-left quadrant and re-map the depth.

In [ ]:
xmin, ymin, xmax, ymax = ds.bounds
mx, my = (xmin + xmax) / 2, (ymin + ymax) / 2
sub = step.subset_by_bounds(xmin, ymin, mx, my)
print("faces in full mesh:", ds.n_face, " -> in subset:", sub.n_face)
_g = sub.plot("Total water depth", cmap="viridis", colorbar=True, title="Total water depth - lower-left quadrant")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The crop keeps only the faces whose geometry falls in the box (803 of 3612) and the field values ride along untouched - a smaller `MeshDataset` that plots, converts and samples exactly like the full one.

## Takeaway

Reading a DFSU needs `mikeio` (the `pyramids-mesh[dhi]` extra); that is the only DFSU-specific step. Once the `dhi` driver has turned DHI's proprietary binary into a `MeshDataset`, everything else - touring the solver fields, mapping surface elevation / depth / velocity at a chosen timestep, plotting the static bathymetry, and the vector/crop conversions - is ordinary pyramids, identical to every other mesh driver. pyramids-mesh owns only the parsing; the analysis lives upstream in pyramids.